# sample7 - Anthropic API（Claude）

**Anthropic API** を使って Claude を Python から呼び出します。  
MCP サーバとの統合に向けた最終ステップです。

| ステップ | 内容 |
|----------|------|
| 1 | 基本的なメッセージ送信 |
| 2 | システムプロンプト |
| 3 | ストリーミング出力 |
| 4 | マルチターン会話 |
| 5 | ツール使用（Function Calling） |

## 事前準備

Anthropic API キーが必要です。

```bash
# ~/.bashrc に追加済みか確認
echo $ANTHROPIC_API_KEY
```

In [ ]:
import anthropic
import os

client = anthropic.Anthropic()  # ANTHROPIC_API_KEY を自動で読み込む
MODEL  = 'claude-opus-4-6'      # 使用するモデル
print("Anthropic クライアント初期化完了")

## 1. 基本的なメッセージ送信

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {'role': 'user', 'content': 'PyTorch を一言で説明してください。'}
    ]
)

print("応答:")
print(message.content[0].text)
print("\n--- メタ情報 ---")
print(f"モデル          : {message.model}")
print(f"入力トークン    : {message.usage.input_tokens}")
print(f"出力トークン    : {message.usage.output_tokens}")

## 2. システムプロンプト

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    system='あなたは機械学習の専門家です。技術的に正確かつ簡潔に回答してください。',
    messages=[
        {'role': 'user', 'content': 'LoRA とは何ですか？'}
    ]
)

print(message.content[0].text)

## 3. ストリーミング出力

In [ ]:
print("応答（ストリーミング）:")

with client.messages.stream(
    model=MODEL,
    max_tokens=256,
    messages=[{'role': 'user', 'content': 'RAG の仕組みを3ステップで説明してください。'}]
) as stream:
    for text in stream.text_stream:
        print(text, end='', flush=True)
print()

## 4. マルチターン会話

会話履歴を messages リストで管理します。

In [ ]:
def chat(messages, user_input, system=None):
    messages.append({'role': 'user', 'content': user_input})
    kwargs = dict(model=MODEL, max_tokens=256, messages=messages)
    if system:
        kwargs['system'] = system
    response = client.messages.create(**kwargs)
    answer = response.content[0].text
    messages.append({'role': 'assistant', 'content': answer})
    return answer

history = []
system  = 'あなたは親切な日本語アシスタントです。'

r1 = chat(history, 'LangChain とは何ですか？', system)
print("1回目:", r1)
print()

r2 = chat(history, 'それはどんな場面で使いますか？', system)  # 文脈を引き継ぐ
print("2回目:", r2)

## 5. ツール使用（Function Calling）

Claude に関数（ツール）を渡し、必要に応じて呼び出させます。  
MCP サーバの基本的な仕組みと同じです。

In [ ]:
import json

# ツールの定義
tools = [
    {
        'name': 'get_model_info',
        'description': '機械学習フレームワークの情報を取得します',
        'input_schema': {
            'type': 'object',
            'properties': {
                'framework': {
                    'type': 'string',
                    'description': 'フレームワーク名（例: PyTorch, TensorFlow）'
                }
            },
            'required': ['framework']
        }
    }
]

# ツールの実装
def get_model_info(framework):
    db = {
        'PyTorch'    : {'開発元': 'Meta',   '言語': 'Python', '特徴': '動的計算グラフ'},
        'TensorFlow' : {'開発元': 'Google', '言語': 'Python', '特徴': '静的計算グラフ'},
        'scikit-learn': {'開発元': 'コミュニティ', '言語': 'Python', '特徴': '古典的ML'},
    }
    return db.get(framework, {'エラー': 'フレームワークが見つかりません'})

# Claude にツールを渡してリクエスト
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    tools=tools,
    messages=[{'role': 'user', 'content': 'PyTorch の情報を教えてください。'}]
)

print("stop_reason:", response.stop_reason)

# ツール呼び出しが発生した場合
if response.stop_reason == 'tool_use':
    tool_use = next(b for b in response.content if b.type == 'tool_use')
    print(f"ツール呼び出し: {tool_use.name}({tool_use.input})")

    # ツールを実行
    result = get_model_info(tool_use.input['framework'])
    print(f"ツール結果: {result}")

    # 結果を Claude に返して最終回答を得る
    final = client.messages.create(
        model=MODEL,
        max_tokens=256,
        tools=tools,
        messages=[
            {'role': 'user', 'content': 'PyTorch の情報を教えてください。'},
            {'role': 'assistant', 'content': response.content},
            {'role': 'user', 'content': [{
                'type': 'tool_result',
                'tool_use_id': tool_use.id,
                'content': json.dumps(result, ensure_ascii=False)
            }]}
        ]
    )
    print("\n最終回答:")
    print(final.content[0].text)